# Prompt Format Diagnostics

Pythia models were pretrained on the Pile (raw internet text) and never saw QA-formatted
prompts during training. This notebook tests multiple prompt formats to determine which
elicits the best numeric generation from Pythia-1.4B, and inspects raw model outputs
to diagnose why exact-match accuracy is 0% on generative tasks.

In [ ]:
import random
import re

import pandas as pd

from sse.models.pythia import PythiaModel

In [ ]:
model = PythiaModel(size="1.4b", dtype="float32")
print(f"Loaded pythia-{model.size} on {model.device}")

## Arithmetic: Compare 3 Prompt Formats

- **Format A** (QA): `"Q: What is 42 + 37?\nA:"` — standard QA, but not in Pile training data
- **Format B** (equation): `"42 + 37 ="` — mathematical notation common in Pile
- **Format C** (compact): `"42+37="` — tightest notation

In [ ]:
rng = random.Random(42)
problems = [(rng.randint(10, 99), rng.randint(10, 99)) for _ in range(20)]

def format_a(a, b):
    return f"Q: What is {a} + {b}?\nA:"

def format_b(a, b):
    return f"{a} + {b} ="

def format_c(a, b):
    return f"{a}+{b}="

formats = {"QA": format_a, "Equation": format_b, "Compact": format_c}

results = []
for name, fmt in formats.items():
    correct = 0
    for a, b in problems:
        prompt = fmt(a, b)
        output = model.generate(prompt, max_new_tokens=10)
        expected = a + b
        match = re.search(r"-?\d+", output)
        got = int(match.group()) if match else None
        is_correct = got == expected
        correct += int(is_correct)
        results.append({
            "format": name, "a": a, "b": b,
            "expected": expected, "got": got,
            "correct": is_correct, "raw_output": output[:60],
        })
    print(f"{name:>10}: {correct}/20 = {correct/20:.0%}")

arith_df = pd.DataFrame(results)

## Raw Output Inspection (Arithmetic)

Look at what the model actually generates for each format:

In [ ]:
for fmt_name in ["QA", "Equation", "Compact"]:
    print(f"\n{'='*60}")
    print(f"Format: {fmt_name}")
    print(f"{'='*60}")
    subset = arith_df[arith_df["format"] == fmt_name].head(10)
    for _, row in subset.iterrows():
        marker = "+" if row["correct"] else "x"
        print(f"  [{marker}] {row['a']}+{row['b']}={row['expected']}"
              f"  got={row['got']}  raw='{row['raw_output']}'")


## TriviaQA: Compare 3 Prompt Formats

- **Format A**: `"Question: {q}\nAnswer:"` — current format
- **Format B**: `"Q: {q}\nA:"` — shorter QA
- **Format C**: `"{q}\nThe answer is"` — completion style

In [ ]:
import datasets

ds = datasets.load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
indices = list(range(len(ds)))
random.Random(42).shuffle(indices)
trivia_items = [ds[i] for i in indices[:20]]

def normalize(text):
    import re as _re
    text = text.lower().strip()
    text = _re.sub(r"[^\w\s]", "", text)
    return " ".join(text.split())

def trivia_format_a(q):
    return f"Question: {q}\nAnswer:"

def trivia_format_b(q):
    return f"Q: {q}\nA:"

def trivia_format_c(q):
    return f"{q}\nThe answer is"

trivia_formats = {
    "Question/Answer": trivia_format_a,
    "Q/A": trivia_format_b,
    "Completion": trivia_format_c,
}

trivia_results = []
for name, fmt in trivia_formats.items():
    correct = 0
    for item in trivia_items:
        question = item["question"]
        aliases = item["answer"]["aliases"]
        prompt = fmt(question)
        output = model.generate(prompt, max_new_tokens=20)
        norm_output = normalize(output)
        is_correct = any(
            normalize(alias) in norm_output
            for alias in aliases if normalize(alias)
        )
        correct += int(is_correct)
        trivia_results.append({
            "format": name, "question": question[:50],
            "aliases": aliases[:3], "correct": is_correct,
            "raw_output": output[:80],
        })
    print(f"{name:>20}: {correct}/20 = {correct/20:.0%}")

trivia_df = pd.DataFrame(trivia_results)

## Raw Output Inspection (TriviaQA)

In [ ]:
for fmt_name in trivia_formats:
    print(f"\n{'='*60}")
    print(f"Format: {fmt_name}")
    print(f"{'='*60}")
    subset = trivia_df[trivia_df["format"] == fmt_name].head(10)
    for _, row in subset.iterrows():
        marker = "+" if row["correct"] else "x"
        print(f"  [{marker}] {row['question']}")
        print(f"       aliases={row['aliases']}")
        print(f"       output='{row['raw_output']}'")
        print()

## Summary

Based on the results above, select the best-performing prompt format for each task.
The winning format is used as the default in the v2 task implementations.

In [ ]:
print("Arithmetic accuracy by format:")
print(arith_df.groupby("format")["correct"].mean().to_string())
print("\nTriviaQA accuracy by format:")
print(trivia_df.groupby("format")["correct"].mean().to_string())

model.unload()
print("\nModel unloaded.")